# Ultrack exp4 revision workflow
Clean release copy of the exp4 notebook. Algorithm cells and parameters are preserved; outputs, GUI/video-only cells and commented alternative experiments are omitted.
The setup uses a NEW output directory and locates the repository from the notebook working directory. It refuses to overwrite archived exp4 outputs. Rerunning is not guaranteed to reproduce IDs deterministically.
Original notebook SHA-256: 67013f8ba1449b951126b5657113c140346ece6c8900ba9539c587d4b63d4d20


In [ ]:
from pathlib import Path
import os
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "revision_code").is_dir() and (p / "livecellx").is_dir())
OUTPUT_DIR = ROOT / "revision_code/reproduced_ultrack_exp4"
OUTPUT_DIR.mkdir(exist_ok=False)
os.chdir(OUTPUT_DIR)


In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = "4"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

from pathlib import Path

import napari
import numpy as np
from napari.utils.notebook_display import nbscreenshot
from tqdm import tqdm
from rich.pretty import pprint
from PIL import Image

from ultrack import track, to_tracks_layer, tracks_to_zarr
from ultrack.imgproc import normalize
from ultrack.utils import estimate_parameters_from_labels, labels_to_contours
from ultrack.utils.array import array_apply
from ultrack.config import MainConfig

In [ ]:
mask_dir = Path(ROOT / "revision_code/data/process2/2D_mask_renamed")
mask_files = sorted(mask_dir.glob("*.png"), key=lambda f: int(f.stem.split("_")[1]))

# read first frame to get spatial dimensions
first = np.array(Image.open(mask_files[0]))
T = len(mask_files)
H, W = first.shape

labels = np.zeros((T, H, W), dtype=np.int32)

for t, f in enumerate(tqdm(mask_files, desc="Loading masks")):
    lab = np.array(Image.open(f))
    labels[t] = lab.astype(np.int32)

print(labels.shape, labels.dtype)  # (579, 1968, 1968), int32

In [ ]:
# convert instance labels to detection (foreground) and edge maps
detection, edges = labels_to_contours(labels, sigma=2.0)

# detection, edges = labels_to_contours(
#     [labels[..., c] for c in range(labels.shape[-1])] +\
#     [labels1[..., c] for c in range(labels1.shape[-1])],
#     sigma=2.0,
# )

# detection, edges = labels_to_contours(
#     [labels, labels1],
#     sigma=2.0,
# )

print(detection.shape, detection.dtype)
print(edges.shape, edges.dtype)

In [ ]:
config = MainConfig()

config.segmentation_config.min_area = 400
config.segmentation_config.max_area = 3000
config.segmentation_config.n_workers = 12
config.segmentation_config.threshold = 0.3
config.segmentation_config.min_frontier = 0.05

config.linking_config.max_distance = 40
config.linking_config.n_workers = 12

config.tracking_config.appear_weight = -4.0
config.tracking_config.disappear_weight = -4.0
config.tracking_config.division_weight = -1
config.tracking_config.power = 1
config.tracking_config.bias = -0.001
config.tracking_config.solution_gap = 0.0

pprint(config)

In [ ]:
track(
    detection=detection,
    edges=edges,
    config=config,
    overwrite=True,
)

In [ ]:
tracks_df, graph = to_tracks_layer(config)



# tracking_labels = tracks_to_zarr(config, tracks_df)

In [ ]:
# export tracking results to zarr
tracking_labels = tracks_to_zarr(
    config,
    tracks_df,
    store_or_path=str(OUTPUT_DIR / "labels.zarr"),
)

In [ ]:
tracks_df.to_csv(OUTPUT_DIR / "tracks_df.csv", index=False)

import json
with open(OUTPUT_DIR / "graph.json", "w") as f:
    json.dump({int(k): int(v) for k, v in graph.items()}, f)